In [ ]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tennie2006push")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Tennie_2006_TenniePush2003.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [ ]:
import pandas as pd
import numpy as np
import pyreadstat


df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)


df['study_id']="tennie2006push"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [ ]:
df.rename(columns={"subject": "ape",
    "species":"species_original",
    "conditio":"condition"}, inplace=True)

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [ ]:
# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

In [ ]:
tennie2006push_standardized=df[['study_id', 'participant', 'sex', 'species', 'condition','case', 'demo', 'do1stany', 'lat1sany'  ]]
comp_out_path_stand = os.path.join(out_pathway, 'tennie2006push_exp2_standardized.csv')
tennie2006push_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =tennie2006push_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
tennie2006push_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tennie2006push_exp2_glossary.csv')
tennie2006push_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

